# OpenRouter eval runner

Drives the LLM/VLM knowledge-tracing eval end-to-end (build windows → call OpenRouter → score). Pure HTTP, single-process — works on a laptop, no GPU required. Iterate parameters in the **Config** cell and re-run from there.

**Prereqs:**
- `export OPENROUTER_API_KEY=sk-or-v1-…` in the shell that launches `jupyter`
- Update `MIAAM_ROOT` below to point at your unpacked MIAAM dataset
- `pip install -r requirements.txt` in your active env

In [ ]:
import os
import subprocess
from pathlib import Path

# Notebook lives in evaluation_llm/; resolve repo root.
EVAL_DIR = Path.cwd().resolve()
if EVAL_DIR.name != "evaluation_llm":
    EVAL_DIR = EVAL_DIR / "evaluation_llm"
REPO_ROOT = EVAL_DIR.parent

# Dataset paths. Update MIAMM_ROOT to wherever the MIAAM dataset is
# unpacked on your machine (parent-of-repo layout by default).
MIAMM_ROOT = REPO_ROOT.parent / "MIAMM"
INTERACTIONS_TEST = REPO_ROOT / "interactions_test.parquet"
DESCRIPTIONS = MIAMM_ROOT / "data/descriptions.json"
SCREENSHOTS_ROOT = MIAMM_ROOT / "data/screenshots/compressed"
EXERCISES_TABLE = MIAMM_ROOT / "data/maths_exercises_table.parquet"

for p in (INTERACTIONS_TEST, DESCRIPTIONS, SCREENSHOTS_ROOT, EXERCISES_TABLE):
    assert p.exists(), f"missing: {p}"

api_key = os.environ.get("OPENROUTER_API_KEY")
assert api_key, "OPENROUTER_API_KEY not set — `export OPENROUTER_API_KEY=sk-or-v1-...` before launching jupyter"
os.environ["OPENAI_API_KEY"] = api_key
print("REPO_ROOT :", REPO_ROOT)
print("EVAL_DIR  :", EVAL_DIR)
print("MIAMM_ROOT:", MIAMM_ROOT)

## Config

Edit and re-run from here downward.

In [ ]:
# ── Task ──
TASK = "prob"           # "prob" (correctness probability) or "answer" (MC option index)

# ── Model & API ──
MODEL = "google/gemma-4-31b-it" # mistralai/mistral-small-2603 google/gemma-4-31b-it openai/gpt-5.5
BASE_URL = "https://openrouter.ai/api/v1"

# ── Eval parameters ──
MODALITY = "text"          # text / vision / both
EXPERT_KNOWLEDGE = False   # augment prompt with activity / objective intents
N_HISTORY = 100            # mistral small only supports 8 images
N_WINDOWS = 4000           # bump for full sweep (e.g. 4000)
CONCURRENCY = 8            # OpenRouter rate-limits per-key; keep modest
MAX_WINDOWS = None         # None = use all from windows.parquet; int = truncate

# ── Generation ──
MAX_TOKENS = 512           # bump (e.g. 4096) for reasoning models
REASONING = "off"          # "on" — bare; "off" — sends chat_template_kwargs={"enable_thinking": False}
REASONING_EFFORT = None    # None / "minimal" — for gpt 5 mini set minimal because None is not supported and defaults to high

# ── Output ──
MODEL_TAG = MODEL.replace("/", "_").replace(":", "_")
PREFIX = "answer_" if TASK == "answer" else ""
RUN_TAG = f"{PREFIX}{MODEL_TAG}_{MODALITY}_n{N_HISTORY}_w{N_WINDOWS}_xk-{int(EXPERT_KNOWLEDGE)}"
OUT_DIR = REPO_ROOT / "results" / RUN_TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)
WINDOWS = OUT_DIR / "windows.parquet"
RESULTS = OUT_DIR / "results.jsonl"
METRICS = OUT_DIR / "metrics.json"

print("Run tag :", RUN_TAG)
print("Out dir :", OUT_DIR)
print("Task    :", TASK)
print("Model   :", MODEL)

## 1. Build windows

Idempotent — skips if `windows.parquet` already exists. Delete it manually to force a rebuild.

In [ ]:
if WINDOWS.exists():
    print(f"[skip] {WINDOWS} already exists")
else:
    script = "build_windows_answer.py" if TASK == "answer" else "build_windows_kt.py"
    cmd = [
        "python", str(EVAL_DIR / script),
        "--input", str(INTERACTIONS_TEST),
        "--output", str(WINDOWS),
        "--descriptions", str(DESCRIPTIONS),
        "--screenshots-root", str(SCREENSHOTS_ROOT),
        "--n-history", str(N_HISTORY),
        "--n-windows", str(N_WINDOWS),
        "--require-modality", MODALITY,
    ]
    if TASK == "answer":
        cmd += ["--exercises-table", str(EXERCISES_TABLE)]
    subprocess.run(cmd, check=True)

## 2. Run eval

Calls OpenRouter for each window. Async, concurrency-bounded. Overwrites `results.jsonl`.

In [ ]:
script = "run_eval_answer.py" if TASK == "answer" else "run_eval_kt.py"
cmd = [
    "python", str(EVAL_DIR / script),
    "--windows", str(WINDOWS),
    "--output", str(RESULTS),
    "--descriptions", str(DESCRIPTIONS),
    "--screenshots-root", str(SCREENSHOTS_ROOT),
    "--modality", MODALITY,
    "--base-url", BASE_URL,
    "--model", MODEL,
    "--max-tokens", str(MAX_TOKENS),
    "--reasoning", REASONING,
    "--concurrency", str(CONCURRENCY),
]
if MAX_WINDOWS:
    cmd += ["--max-windows", str(MAX_WINDOWS)]
if REASONING_EFFORT:
    cmd += ["--reasoning-effort", REASONING_EFFORT]
if EXPERT_KNOWLEDGE:
    cmd += ["--expert-knowledge", "--exercises-table", str(EXERCISES_TABLE)]

subprocess.run(cmd, check=True)

## 3. Score

In [ ]:
script = "score_answer.py" if TASK == "answer" else "score_kt.py"
subprocess.run([
    "python", str(EVAL_DIR / script),
    "--results", str(RESULTS),
    "--windows", str(WINDOWS),
    "--out", str(METRICS),
], check=True)